# Qwen2.5-VL Fragment Step-DPO Training (Kaggle 2xT4 Multi-GPU)

Trains on first-error suffixes (diverging step through Final Answer) from `step_dpo_pairs.jsonl` with prefix masking. Separate from full-trajectory DPO.


In [ ]:
# Cell 1: Check GPU hardware and install sm_60 compatible PyTorch stack if Tesla P100 is assigned
import os, subprocess, sys, torch

print(f'Initial PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    cc = torch.cuda.get_device_capability(0)
    print(f'GPU: {props.name}, Compute Capability: {cc}, VRAM: {props.total_memory / 1e9:.1f} GB')
    if cc[0] < 7:
        print(f'*** Tesla P100 (cc {cc}) detected. PyTorch 2.12 dropped sm_60 CUDA kernels.')
        print('*** Installing PyTorch 2.5.1+cu124 with full sm_60 CUDA GPU support...')
        cmd = [
            sys.executable, '-m', 'pip', 'install', '-q',
            'torch==2.5.1', 'torchvision==0.20.1',
            '--index-url', 'https://download.pytorch.org/whl/cu124',
        ]
        rc = subprocess.run(cmd).returncode
        if rc != 0:
            print('*** Primary torch install failed; fallback: --no-deps + nvidia-cudnn-cu12==9.1.1.17')
            subprocess.run(cmd + ['--no-deps'], check=True)
            subprocess.run([
                sys.executable, '-m', 'pip', 'install', '-q',
                'nvidia-cudnn-cu12==9.1.1.17',
                'nvidia-cublas-cu12', 'nvidia-cuda-runtime-cu12', 'nvidia-cuda-nvrtc-cu12',
                'nvidia-cufft-cu12', 'nvidia-curand-cu12', 'nvidia-cusolver-cu12',
                'nvidia-cusparse-cu12', 'nvidia-nccl-cu12', 'nvidia-nvtx-cu12',
                'triton', 'filelock', 'fsspec', 'jinja2', 'networkx', 'sympy', 'typing-extensions',
            ], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
packages = ['transformers==4.49.0', 'peft==0.14.0', 'accelerate==1.2.1', 'qwen-vl-utils==0.0.14', 'pillow']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)

import torch, transformers
print(f'Active PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Active GPU device: {torch.cuda.get_device_name(0)}')
print('Packages successfully configured.')

In [ ]:
# Cell 2: Checkout repository and execute fragment Step-DPO trainer
import os, subprocess, sys
from pathlib import Path

repo_dir = Path('/tmp/prm_project')
if repo_dir.exists():
    subprocess.run(['rm', '-rf', str(repo_dir)], check=True)

subprocess.run(['git', 'clone', 'https://github.com/yahorlahunovich/prm_project.git', str(repo_dir)], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'fetch', 'origin', 'main'], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'reset', '--hard', 'origin/main'], check=True)
os.chdir(repo_dir)
print(f'Working directory set to {repo_dir}')

env = os.environ.copy()
env['PYTHONPATH'] = 'src'
cmd = [
    sys.executable, 'scripts/train/train_dpo.py',
    '--step-dpo',
    '--output-dir', '/kaggle/working/qwen_vl_step_dpo_adapter',
    '--epochs', '2',
    '--batch-size', '1',
    '--lr', '1e-5',
    '--beta', '0.1',
]
subprocess.run(cmd, env=env, check=True)


In [ ]:
# Cell 3: Validate output artifacts
out_dir = Path('/kaggle/working/qwen_vl_step_dpo_adapter')
files = sorted([p.name for p in out_dir.iterdir()]) if out_dir.exists() else []
print(f'Step-DPO adapter directory {out_dir} contents: {files}')